# MODUL PRAKTIKUM ANALISA BIG DATA (LANJUTAN)
## Fondasi RAG: Pemotongan Teks, Embedding, dan Penyusunan Jawaban

> **Praktikum berbasis Google Colab · Bahasa Indonesia**

---

## Identitas Mahasiswa

| Keterangan | Isi |
|---|---|
| **Nama** | Andreas Rio Christian |
| **NIM** | 2023230031 |
| **Dosen Pengampu** | Herianto, S.Pd., M.T. |
| **Mata Kuliah** | Big Data & Analitik |

---

## 1. Pendahuluan

Ketika berhadapan dengan dokumen yang sangat panjang, kita tidak bisa langsung menyodorkan seluruh isinya ke sebuah Large Language Model (LLM). Ada dua kendala. Pertama, jumlah teks yang dapat diproses model dalam sekali jalan memang dibatasi. Kedua, mengirim teks dalam jumlah besar secara berulang ke layanan LLM komersial membuat biaya komputasinya cepat membengkak.

Pendekatan yang umum dipakai untuk mengatasinya disebut **Retrieval-Augmented Generation**, atau **RAG**. Idenya sederhana. Dokumen yang panjang dipecah dulu menjadi potongan-potongan kecil, lalu setiap potongan diubah menjadi deretan angka (vektor) dan disimpan ke dalam basis data khusus. Begitu pengguna mengajukan pertanyaan, sistem tidak membaca seluruh dokumen, melainkan mencari beberapa potongan yang maknanya paling dekat dengan pertanyaan itu. Potongan itulah yang kemudian diberikan kepada LLM sebagai bahan untuk menyusun jawaban. Dengan cara ini jawaban yang dihasilkan berpijak pada sumber, bukan sekadar tebakan model.

Pada praktikum ini kita akan membangun rangkaian tersebut langkah demi langkah, mulai dari pemotongan teks sampai model menghasilkan jawaban. Dokumen yang dipakai sebagai contoh bersifat fiktif dan hanya untuk keperluan latihan.

## 2. Kompetensi Praktikum

Setelah mengikuti praktikum ini, mahasiswa diharapkan mampu:

- **memotong** dokumen secara rekursif sehingga konteks antarbagian tetap terjaga;
- **mengubah** teks berbahasa Indonesia menjadi vektor dengan bantuan model embedding multilingual yang sudah terlatih;
- **membangun** basis data vektor lokal menggunakan FAISS dan melakukan pencarian semantik di atasnya;
- **menyusun** rangkaian RAG secara utuh sampai model menghasilkan jawaban berdasarkan potongan yang ditemukan.

## 3. Prasyarat dan Catatan Google Colab

Sebelum mulai, perhatikan beberapa hal berikut:

- Anda memerlukan **akun Google** dan akses ke **Google Colab** versi gratis. CPU bawaan sudah memadai untuk praktikum ini; GPU T4 sifatnya opsional dan tidak wajib.
- Jalankan sel kode **berurutan dari atas ke bawah**. Sebagian besar error pada praktikum ini muncul hanya karena ada sel yang terlewat.
- Model embedding berukuran sekitar **480 MB** akan diunduh sekali di awal, biasanya kurang dari satu menit. Setelah itu sistem memakai salinan yang sudah tersimpan.
- Bila tepat setelah proses instalasi muncul error saat mengimpor pustaka, buka menu **Runtime** lalu pilih **Restart session**, kemudian jalankan ulang dari sel pertama. Anda tidak perlu menginstal ulang.
- Langkah 1 sampai 5 berjalan **penuh tanpa API key**. Hanya Langkah 6 yang memerlukan API key gratis dari Groq, dan langkah itu pun sifatnya opsional.

## 4. Menyiapkan Lingkungan di Colab

Sel pertama memasang semua pustaka yang kita perlukan, yaitu kerangka kerja LangChain, model embedding, mesin pencari vektor FAISS, dan konektor ke LLM Groq untuk Langkah 6.

Tanda garis miring terbalik di akhir baris hanya berfungsi menyambung perintah ke baris berikutnya.

In [ ]:
# === Sel 1: Instalasi dependensi (jalankan sekali di awal) ===
%pip install -q langchain-huggingface langchain-community \
    langchain-text-splitters sentence-transformers \
    faiss-cpu langchain-groq

Sebelum melangkah lebih jauh, sebaiknya kita pastikan dulu semua pustaka benar-benar terpasang. Sel berikut hanya mencoba mengimpornya, lalu mencetak satu baris penanda.

In [ ]:
# === Sel 2: Verifikasi instalasi ===
import langchain, faiss, sentence_transformers
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("Semua pustaka inti berhasil di-import. Siap praktikum!")

Bila berhasil, baris terakhir akan mencetak:
```
Semua pustaka inti berhasil di-import. Siap praktikum!
```

> **Catatan:** `HuggingFaceEmbeddings` kini kita ambil dari paket `langchain_huggingface`. Paket lama `langchain_community.embeddings` sebetulnya masih bisa dipakai, tetapi sudah ditandai **usang** dan sebaiknya ditinggalkan.

---

## 5. Langkah-Langkah Praktikum

### Langkah 1: Menyiapkan Korpus Basis Pengetahuan

Pertama-tama kita siapkan sebuah dokumen contoh berbahasa Indonesia yang berperan sebagai sumber pengetahuan. Di sini saya gunakan potongan aturan akademik fiktif. Pada penerapan nyata, dokumen ini bisa Anda ganti dengan panduan atau regulasi yang sesungguhnya.

In [ ]:
# === Sel 3: Basis Pengetahuan (Knowledge Base) ===
dokumen_akademik = """
Panduan Akademik Universitas Darma Persada

Bab 1 - Ketentuan Absensi dan Kelulusan.
Setiap mahasiswa diwajibkan menghadiri perkuliahan minimal 75 persen
dari total pertemuan dalam satu semester. Jika kehadiran kurang dari
75 persen, mahasiswa tidak diperbolehkan mengikuti Ujian Akhir
Semester (UAS).

Bab 2 - Aturan Tugas Akhir dan Skripsi.
Mahasiswa program studi Teknologi Informasi dapat mengajukan proposal
Tugas Akhir setelah menempuh minimal 120 Satuan Kredit Semester (SKS).
Proposal harus disetujui Ketua Program Studi (Kaprodi) dan memiliki
minimal satu dosen pembimbing. Batas waktu penyelesaian skripsi adalah
dua semester sejak proposal disetujui. Jika melewati batas tersebut,
mahasiswa wajib mengajukan perpanjangan atau penggantian topik.
"""
print("Dokumen siap. Total karakter:", len(dokumen_akademik))

**Keluaran yang diharapkan:**
```
Dokumen siap. Total karakter: 723
```

---

### Langkah 2: Memotong Teks (Recursive Chunking)

`RecursiveCharacterTextSplitter` memecah dokumen secara bertahap. Mula-mula ia mencoba memisah pada batas paragraf; kalau potongannya masih terlalu panjang, barulah ia turun ke batas baris, lalu ke spasi. Parameter `chunk_overlap` sengaja kita beri nilai agar ada bagian yang tumpang tindih antarpotongan, supaya kalimat di batas potongan tidak terpotong begitu saja.

In [ ]:
# === Sel 4: Pemotongan teks (Recursive Chunking) ===
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,   # maksimal 150 karakter per fragmen
    chunk_overlap=30  # irisan 30 karakter antar fragmen
)

chunks = text_splitter.split_text(dokumen_akademik)
print(f"Dokumen dipotong menjadi {len(chunks)} bagian.\n")

for i, chunk in enumerate(chunks):
    print(f"--- Chunk ke-{i+1} ({len(chunk)} karakter) ---")
    print(chunk)
    print()

Jumlah potongan bergantung pada panjang dokumen. Untuk contoh ini Anda akan memperoleh sekitar **tujuh potongan**, kurang lebih begini:

```
Dokumen dipotong menjadi 7 bagian.
--- Chunk ke-1 (143 karakter) ---
Panduan Akademik Universitas Darma Persada
Bab 1 - Ketentuan Absensi dan Kelulusan. Setiap mahasiswa diwajibkan ...
...
```

---

### Langkah 3: Mengubah Teks Menjadi Vektor (Embedding)

Berikutnya tiap potongan kita ubah menjadi vektor. Model yang dipakai, `distiluse-base-multilingual-cased-v1`, dipilih karena mendukung banyak bahasa termasuk Indonesia dan cukup ringan untuk dijalankan di CPU.

In [ ]:
# === Sel 5: Transformasi teks menjadi vektor (Embedding) ===
from langchain_huggingface import HuggingFaceEmbeddings

print("Mengunduh model embedding multilingual dari Hugging Face Hub...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/distiluse-base-multilingual-cased-v1"
)
print("Model embedding siap digunakan.")

contoh_vektor = embedding_model.embed_query(chunks[0])
print(f"Dimensi vektor: {len(contoh_vektor)}")
print(f"5 koordinat pertama: {contoh_vektor[:5]}")

Yang penting diperhatikan adalah **dimensinya**, yaitu `512`. Lima angka pertama hanya cuplikan; nilai persisnya tidak harus sama dengan di bawah:

```
Model embedding siap digunakan.
Dimensi vektor: 512
5 koordinat pertama: [0.041, -0.018, 0.067, ...]
```

---

### Langkah 4: Menyimpan ke Basis Data Vektor (FAISS)

Seluruh vektor kemudian kita indeks dengan **FAISS** (Facebook AI Similarity Search). FAISS menyimpan indeksnya langsung di memori, sehingga proses pencarian nanti berlangsung cepat tanpa perlu menyiapkan server tersendiri.

In [ ]:
# === Sel 6: Indexing ke Vector Database (FAISS) ===
from langchain_community.vectorstores import FAISS

print("Membangun indeks FAISS...")
vector_db = FAISS.from_texts(chunks, embedding_model)
print("Vector Database berhasil dibangun.")

**Keluaran yang diharapkan:**
```
Membangun indeks FAISS...
Vector Database berhasil dibangun.
```

---

### Langkah 5: Pencarian Semantik

Sekarang bagian yang paling menarik. Kita ajukan pertanyaan dengan bahasa bebas, dan sistem mencari potongan yang maknanya paling dekat, **bukan yang kata-katanya persis sama**.

Perhatikan misalnya, pertanyaan memakai frasa *"ambil skripsi"*, sedangkan dokumen menulis *"mengajukan proposal Tugas Akhir"*. Meski kata-katanya berbeda, keduanya tetap dikenali berkaitan.

In [ ]:
# === Sel 7: Pencarian semantik (Semantic Search) ===
query = "Berapa SKS syarat untuk ambil skripsi?"
terdekat = vector_db.similarity_search(query, k=2)

print(f"Pertanyaan: {query}\n")
print("Dokumen paling relevan (Top-2):")
print("=" * 60)
for i, doc in enumerate(terdekat):
    print(f"[{i+1}] {doc.page_content}")
    print("-" * 60)

**Hasilnya kira-kira seperti ini:**
```
Pertanyaan: Berapa SKS syarat untuk ambil skripsi?

Dokumen paling relevan (Top-2):
============================================================
[1] ... menempuh minimal 120 Satuan Kredit Semester (SKS) ...
------------------------------------------------------------
```

---

### Langkah 6: Menyusun Jawaban dengan LLM (Generation)

Sampai Langkah 5 sistem baru bisa **menemukan** potongan yang relevan, belum **merangkainya** menjadi jawaban. Pada langkah penutup ini, potongan hasil pencarian kita jadikan konteks, lalu kita serahkan ke sebuah LLM untuk dirumuskan menjadi jawaban yang utuh.

> **Langkah ini membutuhkan API key gratis dari Groq.** Daftar lebih dulu di [console.groq.com](https://console.groq.com), buat sebuah key, lalu tempelkan ketika nanti diminta.

In [ ]:
# === Sel 8 (opsional): Generation - menyusun jawaban ===
import os, getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Masukkan GROQ_API_KEY: ")

from langchain_groq import ChatGroq

# 1) Ambil konteks dari hasil retrieval (Langkah 5)
docs = vector_db.similarity_search(query, k=2)
konteks = "\n\n".join(d.page_content for d in docs)

# 2) Susun prompt yang 'grounded' pada konteks
prompt = f"""Anda asisten akademik. Jawab pertanyaan HANYA berdasarkan konteks.
Jika jawaban tidak ada di dalam konteks, katakan jujur tidak tahu.

Konteks:
{konteks}

Pertanyaan: {query}
Jawaban:"""

# 3) Panggil LLM gratis dari Groq untuk merangkai jawaban
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
jawaban = llm.invoke(prompt)

print("Pertanyaan:", query)
print("\nJawaban:\n", jawaban.content)

Karena model diminta menjawab hanya dari konteks, hasilnya kurang lebih:

```
Pertanyaan: Berapa SKS syarat untuk ambil skripsi?

Jawaban:
Mahasiswa dapat mengajukan proposal Tugas Akhir (skripsi)
setelah menempuh minimal 120 SKS.
```

> **Tip:** Daftar model di Groq sewaktu-waktu bisa berubah. Kalau muncul error `model_decommissioned` atau model tidak ditemukan, buka [console.groq.com/docs/models](https://console.groq.com/docs/models) dan ganti `openai/gpt-oss-20b` dengan model gratis yang sedang tersedia.

---

## 6. Skrip Ringkas Semua Langkah

Bila Anda hanya ingin menguji keseluruhan alur pencarian dengan cepat, lima langkah pertama bisa digabung dalam satu sel. Pastikan sel instalasi sudah dijalankan lebih dulu.

In [ ]:
# === Pipeline RAG (retrieval) dalam satu sel ===
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

dokumen_akademik = """
Panduan Akademik Universitas Darma Persada
Bab 1 - Kehadiran minimal 75 persen; jika kurang, tidak boleh ikut UAS.
Bab 2 - Proposal Tugas Akhir (skripsi) dapat diajukan setelah menempuh
minimal 120 SKS, disetujui Kaprodi, dengan minimal satu pembimbing.
Batas penyelesaian skripsi dua semester sejak proposal disetujui.
"""

# 1) Pemotongan
splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
chunks = splitter.split_text(dokumen_akademik)

# 2) Embedding
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/distiluse-base-multilingual-cased-v1"
)

# 3) Pengindeksan ke FAISS
vector_db = FAISS.from_texts(chunks, embedding_model)

# 4) Pencarian
query = "Berapa SKS syarat untuk ambil skripsi?"
for i, doc in enumerate(vector_db.similarity_search(query, k=2)):
    print(f"[{i+1}] {doc.page_content}\n")

---

## 7. Kendala yang Sering Muncul

Selama mengajar materi ini, beberapa kendala berikut paling sering ditemui. Tabel ini bisa Anda jadikan rujukan cepat saat ada yang tersendat.

| Gejala atau Pesan Error | Yang Perlu Dilakukan |
|---|---|
| `ModuleNotFoundError` / `ImportError` tepat setelah `%pip install` | Buka Runtime, pilih **Restart session**, lalu jalankan ulang dari sel pertama tanpa menginstal ulang. |
| `NameError: dokumen_akademik` / `chunks` / `vector_db is not defined` | Ada sel yang dijalankan tidak berurutan. Jalankan sel-sel sebelumnya dulu, dari atas ke bawah. |
| Proses lama atau seolah *'menggantung'* di Langkah 3 | Ini wajar. Model sekitar 480 MB sedang diunduh untuk pertama kali. Tunggu sampai selesai; eksekusi berikutnya jauh lebih cepat. |
| `SyntaxError` atau muncul tanda kutip yang aneh | Pastikan tanda kutip lurus (`"` dan `'`), bukan versi tipografis. Cara paling aman, jalankan berkas `.ipynb` yang sudah disediakan. |
| `AuthenticationError` atau `401` pada Langkah 6 | API key kosong atau salah. Buat key gratis di [console.groq.com](https://console.groq.com), lalu masukkan saat diminta. |
| `model_decommissioned` atau model tidak ditemukan (Langkah 6) | Nama model di Groq berubah. Gantilah dengan model gratis terbaru dari [console.groq.com/docs/models](https://console.groq.com/docs/models). |
| `numpy` atau `faiss` bermasalah saat diimpor | Buka Runtime, pilih **Restart session**. Bila masih, jalankan ulang sel instalasi lalu restart sekali lagi. |

---

## 8. Tugas dan Evaluasi

Kerjakan ketiga soal berikut sebagai laporan praktikum.

### Soal 1

Ubah `chunk_size` menjadi `50` dan `chunk_overlap` menjadi `0` pada Langkah 2, lalu jalankan ulang. Amati bentuk potongan yang dihasilkan, kemudian jelaskan **kerugian** dari pemotongan yang terlalu pendek terhadap keutuhan makna informasi.

**Jawaban Soal 1:**


In [ ]:
# === Tugas 1: Eksperimen chunk_size kecil ===
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter_kecil = RecursiveCharacterTextSplitter(
    chunk_size=50,    # diperkecil dari 150 menjadi 50
    chunk_overlap=0   # tanpa tumpang tindih
)

chunks_kecil = text_splitter_kecil.split_text(dokumen_akademik)
print(f"Dokumen dipotong menjadi {len(chunks_kecil)} bagian (chunk_size=50, overlap=0):\n")
for i, chunk in enumerate(chunks_kecil):
    print(f"--- Chunk ke-{i+1} ({len(chunk)} karakter) ---")
    print(chunk)
    print()

---

### Soal 2

Ajukan pertanyaan yang sama sekali di luar topik dokumen, misalnya soal cara memasak rendang. Perhatikan potongan apa yang tetap dikembalikan FAISS, lalu terangkan mengapa sistem masih saja mengeluarkan hasil dan apa risikonya bila hasil itu langsung diteruskan ke LLM.

**Jawaban Soal 2:**


In [ ]:
# === Tugas 2: Pertanyaan di luar topik dokumen ===
query_luar_topik = "Bagaimana cara memasak rendang yang enak?"

# Gunakan vector_db dari Langkah 4 (pastikan sudah dijalankan)
hasil_luar_topik = vector_db.similarity_search(query_luar_topik, k=2)

print(f"Pertanyaan: {query_luar_topik}\n")
print("Potongan yang dikembalikan FAISS:")
print("=" * 60)
for i, doc in enumerate(hasil_luar_topik):
    print(f"[{i+1}] {doc.page_content}")
    print("-" * 60)

---

### Soal 3

Pada Langkah 6, coba jalankan dua versi prompt: satu **menyertakan konteks**, satu lagi **tanpa konteks** sama sekali. Bandingkan ketepatan jawaban keduanya, lalu jelaskan peran tahap pencarian dalam menekan jawaban yang mengada-ada.

**Jawaban Soal 3:**


In [ ]:
# === Tugas 3: Perbandingan prompt dengan dan tanpa konteks ===
# Pastikan GROQ_API_KEY sudah diset dari Langkah 6

from langchain_groq import ChatGroq

query_tugas = "Berapa SKS syarat untuk ambil skripsi?"
docs = vector_db.similarity_search(query_tugas, k=2)
konteks = "\n\n".join(d.page_content for d in docs)

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# Versi 1: Dengan konteks
prompt_dengan_konteks = f"""Anda asisten akademik. Jawab pertanyaan HANYA berdasarkan konteks.
Jika jawaban tidak ada di dalam konteks, katakan jujur tidak tahu.

Konteks:
{konteks}

Pertanyaan: {query_tugas}
Jawaban:"""

# Versi 2: Tanpa konteks
prompt_tanpa_konteks = f"""Anda asisten akademik.
Pertanyaan: {query_tugas}
Jawaban:"""

jawaban_dengan_konteks = llm.invoke(prompt_dengan_konteks)
jawaban_tanpa_konteks  = llm.invoke(prompt_tanpa_konteks)

print("=" * 60)
print("VERSI 1 — Dengan Konteks:")
print(jawaban_dengan_konteks.content)
print()
print("=" * 60)
print("VERSI 2 — Tanpa Konteks:")
print(jawaban_tanpa_konteks.content)

---

## 9. Rangkuman

Dari praktikum ini ada beberapa hal yang perlu diingat:

| Tahap | Komponen | Peran |
|---|---|---|
| **Chunking** | `RecursiveCharacterTextSplitter` | Memotong dokumen panjang menjadi fragmen yang bisa diproses |
| **Embedding** | `distiluse-base-multilingual-cased-v1` | Menerjemahkan teks ke vektor angka |
| **Indexing** | FAISS | Menyimpan & mencari vektor secara efisien dalam milidetik |
| **Generation** | ChatGroq / LLM | Merangkai potongan hasil pencarian menjadi jawaban yang utuh |

- Mutu **pemotongan teks** sangat menentukan seberapa utuh konteks yang akhirnya sampai ke model.
- Proses **embedding** berperan sebagai jembatan yang menerjemahkan bahasa manusia ke bentuk angka agar bisa dihitung oleh mesin.
- **FAISS** membuat pencarian pada data yang tidak terstruktur berlangsung dalam hitungan milidetik.
- Tahap **generation** menutup rangkaian itu dengan merangkai potongan hasil pencarian menjadi jawaban yang berpijak pada sumber.

Secara keseluruhan, urutan kerjanya adalah: **memotong → meng-embed → mengindeks → mencari → menyusun jawaban**.

---

> **Pengembangan lanjutan:** Basis pengetahuan dapat diganti dengan dokumen PDF yang sesungguhnya, misalnya panduan akademik resmi, dan dilengkapi antarmuka tanya-jawab sederhana supaya lebih mudah digunakan di kelas.